# Study Analysis: Embodiment and Trust in a Conversational Recommender System

Analysis for the AIES 2026 paper. Data: main study phase-level CSVs (`eca_phase1/2.csv`, `chatbot_phase1/2.csv`).

**Final sample after exclusions: N = 56 (Chatbot = 29, ECA = 27).**

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
# install if needed: pip install factor_analyzer
from factor_analyzer import FactorAnalyzer

pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

covariates_version = "full"

## 1. Load and Merge Phases

In [2]:
DATA_DIR = Path('.')  # all 4 CSVs should sit next to this notebook

csvs = sorted(DATA_DIR.glob('*_phase*.csv'))
assert len(csvs) == 4, f'Expected 4 CSV files, found {len(csvs)}: {[f.name for f in csvs]}'

files = {'eca': {}, 'chatbot': {}}

for f in csvs:
    name_lower = f.name.lower()
    cols = pd.read_csv(f, nrows=0).columns
    phase = 2 if 'perceived_healthiness' in cols else 1 
    cond  = 'eca' if 'eca' in name_lower else 'chatbot' if 'chatbot' in name_lower else None
    assert phase is not None, f'Cannot detect phase for {f.name} — no mp1 or perceived_healthiness column'
    assert cond  is not None, f'Cannot detect condition for {f.name} — filename must contain eca or chatbot'
    files[cond][phase] = f
    print(f'  {f.name:40s} → condition={cond}, phase={phase}')

print()

  chatbot_phase1.csv                       → condition=chatbot, phase=1
  chatbot_phase2.csv                       → condition=chatbot, phase=2
  eca_phase1.csv                           → condition=eca, phase=1
  eca_phase2.csv                           → condition=eca, phase=2



In [3]:
def merge_phases(p1_path, p2_path, condition_label):
    """Load phase 1 + phase 2 CSVs, merge on participant_id, add condition column."""
    p1 = pd.read_csv(p1_path)
    p2 = pd.read_csv(p2_path)
    shared = [c for c in p1.columns if c in p2.columns and c != 'participant_id']
    merged = p1.merge(p2, on='participant_id', how='inner', suffixes=('_p1','_p2'))
    merged['condition'] = condition_label
    only_p1 = set(p1.participant_id) - set(p2.participant_id)
    only_p2 = set(p2.participant_id) - set(p1.participant_id)
    print(f'{condition_label}: {len(p1)} p1 × {len(p2)} p2 → {len(merged)} merged', end='')
    if only_p1 or only_p2:
        print(f'  (dropped: {len(only_p1)} p1-only {only_p1}, {len(only_p2)} p2-only {only_p2})', end='')
    print()
    return merged

eca_df      = merge_phases(files['eca'][1],     files['eca'][2],     'ECA')
chatbot_df  = merge_phases(files['chatbot'][1], files['chatbot'][2], 'Chatbot')

df = chatbot_df  

# Pool all participants — used for psychometric validation and descriptives
df = pd.concat([eca_df, chatbot_df], ignore_index=True)

ECA: 30 p1 × 30 p2 → 30 merged
Chatbot: 30 p1 × 30 p2 → 30 merged


## 2. Exclude Participants

Time-outlier exclusion (±2 SD on completion time), then keep only participants with a recommendation record.

In [4]:
# taking over 2 sd time to complete the study or less than 2 sd is considered too slow or too fast, respectively. This is a common method to identify outliers in time-based data.
time_mean = df['total_duration_seconds_p2'].mean()
time_sd = df['total_duration_seconds_p2'].std()
time_cutoff = time_mean + 2 * time_sd
too_slow = df[df['total_duration_seconds_p2'] > time_cutoff]
print(f'Participants taking >2 SD time to complete: {len(too_slow)}  (mean={time_mean:.1f} min, cutoff={time_cutoff:.1f} min)')     
too_fast = df[df['total_duration_seconds_p2'] < time_mean - 2 * time_sd]
print(f'Participants taking <2 SD time to complete: {len(too_fast)}  (mean={time_mean:.1f} min, cutoff={time_mean - 2 * time_sd:.1f} min)')


# Exclude participants who took too long or too short to complete the study
df = df[(df['total_duration_seconds_p2'] <= time_cutoff) & (df['total_duration_seconds_p2'] >= time_mean - 2 * time_sd)]
print(f'\nAfter excluding outliers based on completion time, remaining N: {len(df)}')   
print(f'  (excluded {len(too_slow)} too slow, {len(too_fast)} too fast)')
print(f' excluded from ECA: {len(too_slow[too_slow.condition=="ECA"]) + len(too_fast[too_fast.condition=="ECA"])} / {len(eca_df)}')
print(f' excluded from Chatbot: {len(too_slow[too_slow.condition=="Chatbot"]) + len(too_fast[too_fast.condition=="Chatbot"])} / {len(chatbot_df)}')

Participants taking >2 SD time to complete: 3  (mean=781.7 min, cutoff=1448.2 min)
Participants taking <2 SD time to complete: 0  (mean=781.7 min, cutoff=115.1 min)

After excluding outliers based on completion time, remaining N: 57
  (excluded 3 too slow, 0 too fast)
 excluded from ECA: 2 / 30
 excluded from Chatbot: 1 / 30


In [5]:
# Make sure Everyone got recommendations and finished the study otherwise exclude

# ── Load ───────────────────────────────────────────────────────────
eca_fsa_summary     = pd.read_csv("eca_fsa_summary.csv")
chatbot_fsa_summary = pd.read_csv("chatbot_fsa_summary.csv")

# ── Filter to only PIDs in df ──────────────────────────────────────
pid_filter = df["participant_id"].unique()
print(f"Filtering FSA summaries to {len(pid_filter)} prolific_pids in df")

eca_fsa_summary     = eca_fsa_summary[eca_fsa_summary["participant_id"].isin(pid_filter)]
chatbot_fsa_summary = chatbot_fsa_summary[chatbot_fsa_summary["participant_id"].isin(pid_filter)]

print(f"After filtering, FSA summary lengths: ECA={len(eca_fsa_summary)}, Chatbot={len(chatbot_fsa_summary)}")


if len(eca_fsa_summary) + len(chatbot_fsa_summary) != len(df):
    df = df[df["participant_id"].isin(eca_fsa_summary["participant_id"].unique()) | df["participant_id"].isin(chatbot_fsa_summary["participant_id"].unique())]
    print(f"After excluding df length is now {len(df)} and FSA summary lengths are ECA={len(eca_fsa_summary)}, Chatbot={len(chatbot_fsa_summary)}")
    assert len(eca_fsa_summary) + len(chatbot_fsa_summary) == len(df), "something went wrong with the merge — check that all prolific_pids in df are in both FSA summaries and that there are no duplicates"

else:
    print("All prolific_pids in df are present in the FSA summaries!")

Filtering FSA summaries to 57 prolific_pids in df
After filtering, FSA summary lengths: ECA=27, Chatbot=29
After excluding df length is now 56 and FSA summary lengths are ECA=27, Chatbot=29


## 3. Demographics

In [6]:
import pandas as pd
import ast

def parse_demographics(demo_value):
    """
    Convert demographics string into dictionary.
    """
    if pd.isna(demo_value):
        return {}

    if isinstance(demo_value, dict):
        return demo_value

    try:
        return ast.literal_eval(demo_value)
    except:
        return {}


def expand_demographics(df, demographics_col="demographics_p2"):
    """
    Expand demographics column into separate columns.
    """
    demo_df = df[demographics_col].apply(parse_demographics).apply(pd.Series)

    result = pd.concat(
        [df.reset_index(drop=True), demo_df.reset_index(drop=True)],
        axis=1
    )

    return result


def demographic_stats_by_condition(
    df,
    demographics_col="demographics_p2",
    condition_col="condition"
):
    """
    Compute absolute counts and percentages
    for demographics per condition.

    Returns:
        dict of DataFrames
    """

    # Expand demographics
    df = expand_demographics(df, demographics_col)

    results = {}

    demographic_columns = ["age", "gender", "cook_frequency"]  # adjust based on your actual demographics

    for col in demographic_columns:

        # Absolute counts
        counts = (
            df.groupby(condition_col)[col]
            .value_counts(dropna=False)
            .unstack(fill_value=0)
        )

        # Percentages
        percentages = (
            counts.div(counts.sum(axis=1), axis=0) * 100
        ).round(2)

        # Combine counts + percentages
        combined = counts.astype(str) + " (" + percentages.astype(str) + "%)"

        results[col] = {
            "counts": counts,
            "percentages": percentages,
            "combined": combined
        }

    return results


def print_demographic_stats(results):
    """
    Pretty print demographic statistics.
    """

    for demographic, stats in results.items():

        print("\n" + "=" * 70)
        print(f"{demographic.upper()} BY CONDITION")
        print("=" * 70)

        print("\nABSOLUTE COUNTS")
        print(stats["counts"])

        print("\nPERCENTAGES")
        print(stats["percentages"])

        print("\nCOMBINED")
        print(stats["combined"])

        print("\n")

results = demographic_stats_by_condition(
    df,
    demographics_col="demographics_p2",
    condition_col="condition"
)

print_demographic_stats(results)


AGE BY CONDITION

ABSOLUTE COUNTS
age        18–24  25–34  35–44  45–54  55–64  65+
condition                                        
Chatbot        7     12      4      4      1    1
ECA            4     14      4      3      2    0

PERCENTAGES
age        18–24  25–34  35–44  45–54  55–64   65+
condition                                         
Chatbot   24.140 41.380 13.790 13.790  3.450 3.450
ECA       14.810 51.850 14.810 11.110  7.410 0.000

COMBINED
age             18–24        25–34       35–44       45–54      55–64  \
condition                                                               
Chatbot    7 (24.14%)  12 (41.38%)  4 (13.79%)  4 (13.79%)  1 (3.45%)   
ECA        4 (14.81%)  14 (51.85%)  4 (14.81%)  3 (11.11%)  2 (7.41%)   

age              65+  
condition             
Chatbot    1 (3.45%)  
ECA         0 (0.0%)  



GENDER BY CONDITION

ABSOLUTE COUNTS
gender     Female  Male
condition              
Chatbot        17    12
ECA            16    11

PERCENTAGES
gend

## 4. Psychometric Validation

EFA loadings, reliability (CR / Cronbach's alpha), and discriminant validity (Fornell–Larcker).

In [7]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='factor_analyzer')

# ── Average T1 and T2 per item for trust scales (more stable validation) ────
# ── behavioral_intention_3 reverse code needed before validation ────────────────────────────────
df['behavioral_intention_3_r'] = (7 + 1) - df['behavioral_intention_3']

# ── Scale definitions ─────────────────────────────────────────────────────────
SCALES = {
    'Goodwill trust':      ['benevolence_trust_1_p1','benevolence_trust_2_p1','benevolence_trust_3_p1','integrity_trust_1_p1','integrity_trust_2_p1','integrity_trust_3_p1'],
    'Qualification trust': ['competence_trust_1_p1','competence_trust_2_p1','competence_trust_3_p1'],
    'Behavioral intention':  ['behavioral_intention_1','behavioral_intention_2','behavioral_intention_3_r'],
    'Continuance intention': ['continuance_intention_1','continuance_intention_2','continuance_intention_3'],
    'Trust disposition':     ['trust_disposition_1','trust_disposition_2','trust_disposition_3'],
    'Familiarity':          ['familiarity_1', 'familiarity_2'],
}

# Retained items carry forward to T1 and T2 composites via _p1/_p2 columns.
# perceived_healthiness is a single item — excluded from EFA/alpha.

LOADING_THRESHOLD = 0.60


def cronbach_alpha(data):
    """Cronbach's alpha from a DataFrame of items."""
    k = data.shape[1]
    if k < 2:
        return np.nan
    item_vars = data.var(ddof=1, axis=0)
    total_var = data.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)


def composite_reliability(loadings):
    """CR = (Σλ)² / ((Σλ)² + Σ(1−λ²))"""
    lam = np.array(loadings)
    return lam.sum()**2 / (lam.sum()**2 + (1 - lam**2).sum())


def ave(loadings):
    """AVE = Σλ² / k"""
    lam = np.array(loadings)
    return (lam**2).mean()


def efa_loadings(data, n_factors=1):
    """Return item loadings from EFA (principal axis, oblimin rotation)."""
    fa = FactorAnalyzer(n_factors=n_factors, rotation='oblimin', method='principal')
    fa.fit(data.dropna())
    result = pd.DataFrame(fa.loadings_, index=data.columns)
    result.columns = [f'factor_{i+1}' for i in range(result.shape[1])]
    return result

# ── First pass: reverse-code behavioral_intention_3 so it exists before validation ──────────────
if 'behavioral_intention_3_r' not in df.columns:
    df['behavioral_intention_3_r'] = (7 + 1) - df['behavioral_intention_3']


# ── Run EFA per scale, flag/drop weak items, report metrics ──────────────────
retained_items   = {}   # scale -> list of retained item columns
dropped_items    = {}   # scale -> list of dropped item columns
validation_rows  = []   # for summary table

print(f'Item dropping threshold: loading < {LOADING_THRESHOLD}\n')
print('=' * 70)

for scale_name, items in SCALES.items():
    # check all items present
    missing_cols = [i for i in items if i not in df.columns]
    if missing_cols:
        print(f'[SKIP] {scale_name} — missing columns: {missing_cols}')
        continue

    data = df[items].dropna()

    if len(items) == 1:
        print(f'{scale_name}: single item — skipping EFA')
        retained_items[scale_name] = items
        continue

    # EFA — 1 factor per scale (confirmatory intent)
    load_df = efa_loadings(data, n_factors=1)
    load_df = load_df.rename(columns={'factor_1': 'loading'})
    load_df['abs_loading'] = load_df['loading'].abs()

    weak = load_df[load_df['abs_loading'] < LOADING_THRESHOLD].index.tolist()
    kept = load_df[load_df['abs_loading'] >= LOADING_THRESHOLD].index.tolist()

    dropped_items[scale_name]  = weak
    retained_items[scale_name] = kept

    if weak:
        print(f'[DROP] {scale_name}: dropped {weak} (loading < {LOADING_THRESHOLD})')

    if len(kept) < 2:
        print(f'  WARNING: only {len(kept)} item(s) retained — cannot compute alpha/CR/AVE reliably')
        validation_rows.append({'Scale': scale_name, 'N items': len(kept),
                                 'alpha': np.nan, 'CR': np.nan, 'AVE': np.nan,
                                 'Dropped': str(weak)})
        continue

    kept_data = df[kept].dropna()
    # refit EFA on retained items to get final loadings
    load_final = efa_loadings(kept_data, n_factors=1)['factor_1'].abs().values

    alpha = cronbach_alpha(kept_data)
    cr    = composite_reliability(load_final)
    av    = ave(load_final)

    validation_rows.append({
        'Scale':    scale_name,
        'N items':  len(kept),
        'alpha':    round(alpha, 3),
        'CR':       round(cr, 3),
        'AVE':      round(av, 3),
        'sqrt_AVE': round(np.sqrt(av), 3),
        'Dropped':  str(weak) if weak else '—',
    })

print('=' * 70)
val_df = pd.DataFrame(validation_rows).set_index('Scale')
print('\nValidation summary (Table A1):')
display(val_df)

# Flag scales below thresholds
issues = val_df[(val_df['alpha'] < 0.70) | (val_df['CR'] < 0.70) | (val_df['AVE'] < 0.50)]
if not issues.empty:
    print('\n⚠ Scales below recommended thresholds (α/.CR < .70 or AVE < .50):')
    display(issues)
else:
    print('\n✓ All scales meet α ≥ .70, CR ≥ .70, AVE ≥ .50')


for scale_name, items in SCALES.items():
    if len(items) < 2:
        continue
    data = df[items].dropna()
    fa = FactorAnalyzer(n_factors=1, rotation='oblimin', method='principal')
    fa.fit(data)
    loadings = pd.Series(fa.loadings_[:,0], index=items)
    print(f"\n{scale_name}:")
    print(loadings.abs().round(2).to_string())

Item dropping threshold: loading < 0.6


Validation summary (Table A1):


/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: '

,N items,alpha,CR,AVE,sqrt_AVE,Dropped
Scale,,,,,,
Goodwill trust,6,0.911,0.937,0.714,0.845,—
Qualification trust,3,0.961,0.975,0.928,0.963,—
Behavioral intention,3,0.886,0.933,0.824,0.908,—
Continuance intention,3,0.975,0.984,0.953,0.976,—
Trust disposition,3,0.823,0.896,0.742,0.861,—
Familiarity,2,0.674,0.872,0.773,0.879,—



⚠ Scales below recommended thresholds (α/.CR < .70 or AVE < .50):


,N items,alpha,CR,AVE,sqrt_AVE,Dropped
Scale,,,,,,
Familiarity,2,0.674,0.872,0.773,0.879,—



Goodwill trust:
benevolence_trust_1_p1   0.890
benevolence_trust_2_p1   0.840
benevolence_trust_3_p1   0.820
integrity_trust_1_p1     0.880
integrity_trust_2_p1     0.840
integrity_trust_3_p1     0.790

Qualification trust:
competence_trust_1_p1   0.970
competence_trust_2_p1   0.960
competence_trust_3_p1   0.960

Behavioral intention:
behavioral_intention_1     0.960
behavioral_intention_2     0.960
behavioral_intention_3_r   0.790

Continuance intention:
continuance_intention_1   0.970
continuance_intention_2   0.980
continuance_intention_3   0.980

Trust disposition:
trust_disposition_1   0.880
trust_disposition_2   0.920
trust_disposition_3   0.790

Familiarity:
familiarity_1   0.880
familiarity_2   0.880


/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: '

In [8]:
# ── Cross-loadings table ──────────────────────────────────────────────────────
# Stack all retained items and run multi-factor EFA to check cross-loadings

all_retained = [item for items in retained_items.values() for item in items]
n_factors    = len([s for s in retained_items if len(retained_items[s]) >= 2])

efa_all = FactorAnalyzer(n_factors=min(n_factors, len(all_retained)-1),
                          rotation='oblimin', method='principal')
efa_all.fit(df[all_retained].dropna())

cross_df = pd.DataFrame(
    efa_all.loadings_,
    index=all_retained,
    columns=[f'F{i+1}' for i in range(efa_all.loadings_.shape[1])]
).round(3)

# Highlight: each item's highest loading
def highlight_max(row):
    styles = ['' for _ in row]
    max_idx = row.abs().argmax()
    styles[max_idx] = 'font-weight: bold; color: #1a5276'
    # flag if any cross-loading abs > intended loading abs
    return styles

print('Cross-loading matrix (bold = highest absolute loading per item):')
display(cross_df.style.apply(highlight_max, axis=1))

Cross-loading matrix (bold = highest absolute loading per item):


/Users/halimeh/anaconda3/envs/agents/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


,F1,F2,F3,F4,F5,F6
benevolence_trust_1_p1,0.550000,0.142000,0.113000,-0.022000,0.368000,-0.205000
benevolence_trust_2_p1,0.407000,0.226000,0.056000,0.125000,0.277000,-0.435000
benevolence_trust_3_p1,0.257000,-0.207000,-0.007000,0.236000,0.664000,-0.051000
integrity_trust_1_p1,0.075000,0.259000,0.153000,-0.107000,0.733000,0.007000
integrity_trust_2_p1,-0.135000,0.131000,-0.007000,0.026000,0.903000,-0.045000
integrity_trust_3_p1,0.212000,-0.103000,-0.058000,-0.006000,0.784000,0.096000
competence_trust_1_p1,0.903000,-0.042000,-0.006000,0.166000,0.009000,0.003000
competence_trust_2_p1,0.899000,0.081000,0.065000,0.014000,0.004000,0.053000
competence_trust_3_p1,0.886000,0.089000,-0.082000,-0.026000,0.078000,0.032000
behavioral_intention_1,0.060000,0.864000,0.017000,-0.032000,0.040000,0.209000


In [9]:
# ── Discriminant validity: Fornell-Larcker criterion ─────────────────────────
# √AVE of each construct must exceed its correlation with every other construct

# Build composite scores from retained items for correlation matrix
temp_composites = {}
for scale, items in retained_items.items():
    if len(items) >= 1:
        temp_composites[scale] = df[items].mean(axis=1)

comp_df = pd.DataFrame(temp_composites)
corr_matrix = comp_df.corr().round(3)

# √AVE per scale
sqrt_ave = val_df['sqrt_AVE'].to_dict()

print('Fornell-Larcker criterion:')
print('√AVE (diagonal) must exceed all off-diagonal correlations in its row/column\n')

fl_matrix = corr_matrix.copy()
for scale in fl_matrix.index:
    if scale in sqrt_ave:
        fl_matrix.loc[scale, scale] = sqrt_ave[scale]  # replace diagonal with √AVE

# Flag violations
violations = []
scales_list = list(fl_matrix.index)
for i, s1 in enumerate(scales_list):
    for j, s2 in enumerate(scales_list):
        if i >= j:
            continue
        corr_val = abs(corr_matrix.loc[s1, s2])
        sv1 = sqrt_ave.get(s1, np.nan)
        sv2 = sqrt_ave.get(s2, np.nan)
        if corr_val > sv1 or corr_val > sv2:
            violations.append(f'  {s1} vs {s2}: r={corr_val:.3f}, √AVE=({sv1:.3f}, {sv2:.3f})')

display(fl_matrix.style.background_gradient(cmap='Blues', axis=None))

if violations:
    print('\n⚠ Fornell-Larcker violations (r > √AVE):')
    for v in violations:
        print(v)
else:
    print('\n✓ Discriminant validity confirmed — all √AVE > inter-construct correlations')
    max_corr = corr_matrix.where(~np.eye(len(corr_matrix), dtype=bool)).abs().max().max()
    print(f'  Highest inter-construct correlation: r = {max_corr:.3f}')

Fornell-Larcker criterion:
√AVE (diagonal) must exceed all off-diagonal correlations in its row/column



,Goodwill trust,Qualification trust,Behavioral intention,Continuance intention,Trust disposition,Familiarity
Goodwill trust,0.845000,0.742000,0.404000,0.564000,0.549000,0.307000
Qualification trust,0.742000,0.963000,0.561000,0.563000,0.563000,0.132000
Behavioral intention,0.404000,0.561000,0.908000,0.824000,0.486000,0.022000
Continuance intention,0.564000,0.563000,0.824000,0.976000,0.546000,0.103000
Trust disposition,0.549000,0.563000,0.486000,0.546000,0.861000,0.331000
Familiarity,0.307000,0.132000,0.022000,0.103000,0.331000,0.879000



✓ Discriminant validity confirmed — all √AVE > inter-construct correlations
  Highest inter-construct correlation: r = 0.824


## 5. Manipulation Check (FSA)

In [10]:
# ── T-tests comparing ECA vs Chatbot on healthy FSA, unhealthy FSA, and distance ─── 

from scipy import stats


# ── Distance (unhealthy - healthy) per system ──────────────────────
eca_fsa_summary["distance"]     = eca_fsa_summary["avg_fsa_unhealthy"]     - eca_fsa_summary["avg_fsa_healthy"]
chatbot_fsa_summary["distance"] = chatbot_fsa_summary["avg_fsa_unhealthy"] - chatbot_fsa_summary["avg_fsa_healthy"]

print(f"ECA participants:     {len(eca_fsa_summary)}")
print(f"Chatbot participants: {len(chatbot_fsa_summary)}\n")

# ── T-test 1: healthy FSA — eca vs chatbot ────────────────────────
t1, p1 = stats.ttest_ind(eca_fsa_summary["avg_fsa_healthy"], chatbot_fsa_summary["avg_fsa_healthy"])
print("── Healthy FSA: ECA vs Chatbot ──────────────────────────────")
print(f"  ECA mean:     {eca_fsa_summary['avg_fsa_healthy'].mean():.3f}")
print(f"  Chatbot mean: {chatbot_fsa_summary['avg_fsa_healthy'].mean():.3f}")
print(f"  t={t1:.3f}, p={p1:.4f}\n")

# ── T-test 2: unhealthy FSA — eca vs chatbot ─────────────────────
t2, p2 = stats.ttest_ind(eca_fsa_summary["avg_fsa_unhealthy"], chatbot_fsa_summary["avg_fsa_unhealthy"])
print("── Unhealthy FSA: ECA vs Chatbot ────────────────────────────")
print(f"  ECA mean:     {eca_fsa_summary['avg_fsa_unhealthy'].mean():.3f}")
print(f"  Chatbot mean: {chatbot_fsa_summary['avg_fsa_unhealthy'].mean():.3f}")
print(f"  t={t2:.3f}, p={p2:.4f}\n")

# ── T-test 3: distance (unhealthy - healthy) — eca vs chatbot ────
t3, p3 = stats.ttest_ind(eca_fsa_summary["distance"], chatbot_fsa_summary["distance"])
print("── Distance (unhealthy − healthy): ECA vs Chatbot ───────────")
print(f"  ECA mean:     {eca_fsa_summary['distance'].mean():.3f}")
print(f"  Chatbot mean: {chatbot_fsa_summary['distance'].mean():.3f}")
print(f"  t={t3:.3f}, p={p3:.4f}\n")


ECA participants:     27
Chatbot participants: 29

── Healthy FSA: ECA vs Chatbot ──────────────────────────────
  ECA mean:     5.185
  Chatbot mean: 5.287
  t=-0.650, p=0.5187

── Unhealthy FSA: ECA vs Chatbot ────────────────────────────
  ECA mean:     10.963
  Chatbot mean: 10.943
  t=0.177, p=0.8604

── Distance (unhealthy − healthy): ECA vs Chatbot ───────────
  ECA mean:     5.778
  Chatbot mean: 5.655
  t=0.635, p=0.5278



## 6. Descriptive Statistics by Condition

In [11]:
def avg(cols):
    return df[cols].mean(axis=1)

# Reverse-code behavioral_intention_3 (7-point scale: 8 - x)
df['behavioral_intention_3_r'] = 8 - df['behavioral_intention_3']

# Phase 1 composites (phase-specific items have no suffix; shared items get _p1)
df['social_presence']          = avg(['social_presence_3', 'social_presence_1', 'social_presence_4', 'social_presence_5', 'social_presence_2'])
df['affect_trust_p1']          = avg(['affect_trust_1_p1','affect_trust_2_p1',])
df['cognition_trust_p1']       = avg(['integrity_trust_1_p1','integrity_trust_2_p1','integrity_trust_3_p1','benevolence_trust_1_p1','benevolence_trust_2_p1','benevolence_trust_3_p1','competence_trust_1_p1','competence_trust_2_p1','competence_trust_3_p1'])
df['overall_affect_cognition_p1']  = avg(['affect_trust_1_p1','affect_trust_2_p1','integrity_trust_1_p1','integrity_trust_2_p1','integrity_trust_3_p1','benevolence_trust_1_p1','benevolence_trust_2_p1','benevolence_trust_3_p1','competence_trust_1_p1','competence_trust_2_p1','competence_trust_3_p1'])


df['goodwill_trust_p1']        = avg(['benevolence_trust_1_p1', 'benevolence_trust_2_p1', 'benevolence_trust_3_p1', 'integrity_trust_1_p1', 'integrity_trust_2_p1', 'integrity_trust_3_p1'])
df['competence_p1']            = avg(['competence_trust_1_p1', 'competence_trust_2_p1', 'competence_trust_3_p1'])
df['overall_goodwill_qual_p1']    = avg(['benevolence_trust_1_p1', 'benevolence_trust_2_p1', 'benevolence_trust_3_p1', 'integrity_trust_1_p1', 'integrity_trust_2_p1', 'integrity_trust_3_p1', 'competence_trust_1_p1', 'competence_trust_2_p1', 'competence_trust_3_p1'])

# Phase 2 composites
df['health_rating']            = df['perceived_healthiness']
df['affect_trust_p2']          = avg(['affect_trust_1_p2', 'affect_trust_2_p2'])
df['cognition_trust_p2']       = avg(['integrity_trust_1_p2','integrity_trust_2_p2','integrity_trust_3_p2','benevolence_trust_1_p2','benevolence_trust_2_p2','benevolence_trust_3_p2','competence_trust_1_p2','competence_trust_2_p2','competence_trust_3_p2'])
df['overall_affect_cognition_p2']  = avg(['affect_trust_1_p2','affect_trust_2_p2','integrity_trust_1_p2','integrity_trust_2_p2','integrity_trust_3_p2','benevolence_trust_1_p2','benevolence_trust_2_p2','benevolence_trust_3_p2','competence_trust_1_p2','competence_trust_2_p2','competence_trust_3_p2'])


df['goodwill_trust_p2']        = avg(['benevolence_trust_1_p2', 'benevolence_trust_2_p2', 'benevolence_trust_3_p2', 'integrity_trust_1_p2', 'integrity_trust_2_p2', 'integrity_trust_3_p2'])
df['competence_p2']            = avg(['competence_trust_1_p2', 'competence_trust_2_p2', 'competence_trust_3_p2'])
df['overall_goodwill_qual_p2']    = avg(['benevolence_trust_1_p2', 'benevolence_trust_2_p2', 'benevolence_trust_3_p2', 'integrity_trust_1_p2', 'integrity_trust_2_p2', 'integrity_trust_3_p2', 'competence_trust_1_p2', 'competence_trust_2_p2', 'competence_trust_3_p2'])

df['behavioral_intention']     = avg(['behavioral_intention_1', 'behavioral_intention_2', 'behavioral_intention_3_r'])
df['continuance_intention']    = avg(['continuance_intention_1', 'continuance_intention_2', 'continuance_intention_3'])

df['health_consciousness']     = avg(['health_consciousness_1', 'health_consciousness_2', 'health_consciousness_3'])
df['trust_disposition']        = avg(['trust_disposition_1', 'trust_disposition_2', 'trust_disposition_3'])
df['familiarity']              = avg(['familiarity_1', 'familiarity_2'])


# Deltas (T1 - T2: positive = trust dropped)
df['delta_affect']        = df['affect_trust_p1']    - df['affect_trust_p2']
df['delta_cognition']     = df['cognition_trust_p1'] - df['cognition_trust_p2']
df['delta_overall_affect_cognition']     = df['overall_affect_cognition_p1'] - df['overall_affect_cognition_p2']


df['delta_goodwill']      = df['goodwill_trust_p1']  - df['goodwill_trust_p2']
df['delta_competence']    = df['competence_p1']      - df['competence_p2']
df['delta_overall_goodwill_qual']       = df['overall_goodwill_qual_p1']    - df['overall_goodwill_qual_p2']


# Percentage trust loss per participant: (T1 - T2) / T1 * 100  (positive = trust dropped)
df['pct_loss_competence']  = (df['competence_p1']      - df['competence_p2'])      / df['competence_p1']      * 100
df['pct_loss_goodwill']    = (df['goodwill_trust_p1']  - df['goodwill_trust_p2'])  / df['goodwill_trust_p1']  * 100
df['pct_loss_overall_goodwill_qual'] = (df['overall_goodwill_qual_p1'] - df['overall_goodwill_qual_p2']) / df['overall_goodwill_qual_p1'] * 100

COMPOSITES = [
    'social_presence',
    'health_rating',
    'health_consciousness', 
    'trust_disposition', 
    'goodwill_trust_p1', 
    'competence_p1',
    'overall_goodwill_qual_p1',
    'goodwill_trust_p2', 
    'competence_p2',
    'overall_goodwill_qual_p2',
    'delta_goodwill', 
    'delta_competence',
    'delta_overall_goodwill_qual',
    'pct_loss_goodwill', 
    'pct_loss_competence',
    'pct_loss_overall_goodwill_qual',
    'behavioral_intention', 
    'continuance_intention',
    'familiarity',
]

CONDITION_COL = 'condition'
ALPHA = 0.05

def sig_stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < ALPHA else 'n.s.'

outcome_vars = COMPOSITES

# p1 baseline for each variable — used as denominator for the % diff column.
# p2 and delta variables → their corresponding p1; everything else → itself.
var_to_p1 = {
    'affect_trust_p2':    'affect_trust_p1',
    'cognition_trust_p2': 'cognition_trust_p1',
    'overall_affect_cognition_p2': 'overall_affect_cognition_p1',
    'goodwill_trust_p2':  'goodwill_trust_p1',
    'competence_p2':      'competence_p1',
    'overall_goodwill_qual_p2': 'overall_goodwill_qual_p1',
    'delta_affect':       'affect_trust_p1',
    'delta_cognition':    'cognition_trust_p1',
    'delta_overall_affect_cognition': 'overall_affect_cognition_p1',
    'delta_goodwill':     'goodwill_trust_p1',
    'delta_competence':   'competence_p1',
    'delta_overall_goodwill_qual': 'overall_goodwill_qual_p1',
    'pct_loss_goodwill':   'goodwill_trust_p1',
    'pct_loss_competence': 'competence_p1',
    'pct_loss_overall_goodwill_qual': 'overall_goodwill_qual_p1',
}

# ── Build means table ────────────────────────────────────────────────────────

means = (
    df.groupby(CONDITION_COL)[outcome_vars]
    .agg(['mean', 'std'])
    .round(3)
    .T
)

pct_diff = {}
for v in outcome_vars:
    p1_col   = var_to_p1.get(v, v)
    p1_mean  = df[p1_col].mean()
    pct_diff[v] = (means.loc[v, 'ECA'] - means.loc[v, 'Chatbot']) 


# ── Highlight higher condition ────────────────────────────────────────────────
def highlight_higher(row):
    styles = [''] * len(row)
    cols = row.index.tolist()
    if 'ECA' in cols and 'Chatbot' in cols:
        ei = cols.index('ECA'); ci = cols.index('Chatbot')
        if row['ECA'] > row['Chatbot']:
            styles[ei] = 'background-color: #c6efce; color: #276221; font-weight: bold'
        elif row['Chatbot'] > row['ECA']:
            styles[ci] = 'background-color: #c6efce; color: #276221; font-weight: bold'
    return styles

print('Means by condition  |  % diff = (ECA - Chatbot) / p1 baseline  |  green = higher')
display(means.style
    .apply(highlight_higher, axis=1)
    .format({'Chatbot': '{:.3f}', 'ECA': '{:.3f}', '% diff (ECA-Chatbot) / p1': '{:+.1f}%'})
)



Means by condition  |  % diff = (ECA - Chatbot) / p1 baseline  |  green = higher


## 7. Covariates and Randomisation Checks

In [12]:
import ast

df['demographics_p2'] = df['demographics_p2'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df = pd.concat([df.drop(columns=['demographics_p2']),
                df['demographics_p2'].apply(pd.Series)], axis=1)

In [13]:
from scipy.stats import chi2_contingency

for var in ['age', 'gender', 'cook_frequency', ]:
    ct = pd.crosstab(df['condition'], df[var])
    chi2, p, dof, expected = chi2_contingency(ct)
    print(f"{var:<20}: chi2 = {chi2:.3f}, df = {dof}, p = {p:.3f} {'*' if p<.05 else 'n.s.'}")

age                 : chi2 = 2.380, df = 5, p = 0.794 n.s.
gender              : chi2 = 0.000, df = 1, p = 1.000 n.s.
cook_frequency      : chi2 = 3.304, df = 4, p = 0.508 n.s.


## 8. Covariate Screening

In [14]:
import statsmodels.formula.api as smf



def covariate_screening(data, covariates, outcomes):
    """
    Tests each covariate against each outcome variable.
    Prints b, p, and significance for every combination.
    Use results to justify covariate inclusion decisions.
    """
    print("\n" + "="*60)
    print("COVARIATE SCREENING")
    print("="*60)
    print(f"{'Covariate':<25} {'Outcome':<30} {'b':>7} {'p':>7} {'sig':<10}")
    print("-"*75)

    for cov in covariates:
        for outcome in outcomes:
            try:
                m   = smf.ols(f'{outcome} ~ {cov}', data=data).fit()
                key = [k for k in m.params.index if k != 'Intercept'][0]
                b   = m.params[key]
                p   = m.pvalues[key]
                sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
                print(f"  {cov:<23} {outcome:<30} {b:>7.3f} {p:>7.3f} {sig}")
            except Exception as e:
                print(f"  {cov:<23} {outcome:<30} ERROR: {e}")
        print()


# ── Run it ────────────────────────────────────────────────────

covariates = [
    'trust_disposition',
    'familiarity',
    'age',
    'gender',
    'cook_frequency',
]

outcomes = [
    'delta_goodwill',
    'delta_competence',
    'delta_overall_goodwill_qual',
    'behavioral_intention',
    'continuance_intention',
    'health_rating',
]


covariate_screening(df, covariates, outcomes)


COVARIATE SCREENING
Covariate                 Outcome                              b       p sig       
---------------------------------------------------------------------------
  trust_disposition       delta_goodwill                  -0.197   0.066 n.s.
  trust_disposition       delta_competence                -0.082   0.638 n.s.
  trust_disposition       delta_overall_goodwill_qual     -0.159   0.190 n.s.
  trust_disposition       behavioral_intention             0.803   0.000 ***
  trust_disposition       continuance_intention            0.960   0.000 ***
  trust_disposition       health_rating                    0.580   0.004 **

  familiarity             delta_goodwill                   0.071   0.547 n.s.
  familiarity             delta_competence                 0.200   0.290 n.s.
  familiarity             delta_overall_goodwill_qual      0.114   0.388 n.s.
  familiarity             behavioral_intention             0.040   0.871 n.s.
  familiarity             continuance_inte

## 9. Hypothesis Tests (with Covariates)

In [15]:
""" 
Full Analysis Script — Configurable Trust Variables

TRUST_GOODWILL : the "warm/relational" trust dimension (H1a, mediator in H3a/H3b)
TRUST_QUAL     : the "competence/qualification" trust dimension (H2, second mediator in H3b)
"""

# ══════════════════════════════════════════════════════════════════════
# ▶▶  SET TRUST VARIABLES HERE
TRUST_GOODWILL = 'delta_goodwill'    # warm/relational trust dimension
TRUST_QUAL     = 'delta_competence'  # competence/qualification trust dimension
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

# ─────────────────────────────────────────────────────────────
# 0. PREPARE VARIABLES
# ─────────────────────────────────────────────────────────────

df['cond_dummy'] = (df['condition'] == 'ECA').astype(int)

# Overall trust = average of the two chosen dimensions
df['_delta_overall']   = df[[TRUST_GOODWILL, TRUST_QUAL]].mean(axis=1)
df['_trust_goodwill']  = df[TRUST_GOODWILL].copy()
df['_trust_qual']      = df[TRUST_QUAL].copy()

# Friendly labels for output
LABEL_OVERALL  = f'overall ({TRUST_GOODWILL} + {TRUST_QUAL})'
LABEL_GOODWILL = TRUST_GOODWILL
LABEL_QUAL     = TRUST_QUAL

# Internal variable names (always fixed so formulas don't change)
VAR_OVERALL  = '_delta_overall'
VAR_GOODWILL = '_trust_goodwill'
VAR_QUAL     = '_trust_qual'

# ─────────────────────────────────────────────────────────────
# COVARIATE MAP  (update if screening results change)
# ─────────────────────────────────────────────────────────────
#'health_consciousness',
COVS = {
    VAR_OVERALL  : ['trust_disposition', 'age', 'gender', 'familiarity', 'cook_frequency'],   
    VAR_GOODWILL : [ 'trust_disposition', 'age',  'gender', 'familiarity', 'cook_frequency'],
    VAR_QUAL     : [  'trust_disposition', 'age', 'gender', 'familiarity', 'cook_frequency'],
    'health_rating'        : [ 'trust_disposition', 'age',  'gender', 'familiarity', 'cook_frequency'],
    'behavioral_intention' : [ 'trust_disposition', 'age',  'gender', 'familiarity', 'cook_frequency'],
    'continuance_intention': [ 'trust_disposition', 'age', 'gender', 'familiarity', 'cook_frequency'],
}

ALL_OUTCOMES = ['health_rating', 'behavioral_intention', 'continuance_intention']

# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────

results_log = []

def cov_str(key):
    covs = COVS.get(key, [])
    return (' + ' + ' + '.join(covs)) if covs else ''

def sig_stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'n.s.'

def section(title, subtitle=''):
    print("\n" + "=" * 70)
    print(f"  {title}")
    if subtitle:
        print(f"  {subtitle}")
    print("=" * 70)

def subsection(title):
    print(f"\n  {'─' * 60}")
    print(f"  {title}")
    print(f"  {'─' * 60}")

def print_result(label, b, p, r2=None, f=None, log_key=None):
    sig = sig_stars(p)
    print(f"\n  Testing  : {label}")
    print(f"  b = {b:+.3f}   p = {p:.3f}   {sig}")
    if r2 is not None:
        print(f"  R² = {r2:.3f}   F = {f:.2f}")
    print(f"  Result   : {'SUPPORTED ✓' if p < .05 else 'NOT SUPPORTED ✗'}")
    if log_key:
        results_log.append({
            'Test': log_key, 'b': round(b, 3), 'p': round(p, 3),
            'sig': sig, 'R2': round(r2, 3) if r2 is not None else np.nan,
            'CI': '—', 'Supported': p < .05
        })

def run_ols(formula, data, pred_key, label, log_key=None):
    m  = smf.ols(formula, data=data).fit()
    b  = m.params[pred_key]
    p  = m.pvalues[pred_key]
    r2 = m.rsquared
    f  = m.fvalue
    f_p = m.f_pvalue
    
    # 95% CI for the predictor of interest
    ci  = m.conf_int().loc[pred_key]
    ci_lo, ci_hi = ci[0], ci[1]
    
    # Partial f² for predictor of interest
    # Run reduced model (without pred_key)
    formula_reduced = formula.replace(f'+ {pred_key}', '').replace(f'{pred_key} +', '').replace(pred_key, '')
    # Clean up double spaces or leading/trailing operators
    import re
    formula_reduced = re.sub(r'\s+', ' ', formula_reduced).strip()
    m_reduced = smf.ols(formula_reduced, data=data).fit()
    r2_reduced = m_reduced.rsquared
    partial_f2 = (r2 - r2_reduced) / (1 - r2) if r2 < 1 else float('inf')
    
    print(f"\n  Formula     : {formula}")
    print(f"  b = {b:.3f}   95% CI [{ci_lo:.3f}, {ci_hi:.3f}]   p = {p:.3f}   "
          f"partial f² = {partial_f2:.3f}   R² = {r2:.3f}   "
          f"F({int(m.df_model)}, {int(m.df_resid)}) = {f:.2f}, p = {f_p:.3f}")
    
    print_result(label, b, p, r2, f, log_key=log_key)
    return m, partial_f2, ci_lo, ci_hi

def bootstrap_mediation(data, x, m, y, covs=None, n_boot=5000, ci=95, label="", log_key=None):
    """Simple mediation — PROCESS Model 4: x -> m -> y"""
    np.random.seed(42)
    indirect = []
    cs = (' + ' + ' + '.join(covs)) if covs else ''
    for _ in range(n_boot):
        s = data.sample(len(data), replace=True)
        a = smf.ols(f'{m} ~ {x}', data=s).fit().params[x]
        b = smf.ols(f'{y} ~ {m} + {x}{cs}', data=s).fit().params[m]
        indirect.append(a * b)
    indirect = np.array(indirect)
    alpha_pct = (100 - ci) / 2
    lo, hi = np.percentile(indirect, [alpha_pct, 100 - alpha_pct])
    est = np.mean(indirect)
    supported = not (lo <= 0 <= hi)
    sig = "SUPPORTED ✓" if supported else "NOT SUPPORTED ✗ (CI includes 0)"
    print(f"\n  Testing  : {label}")
    print(f"  Model    : PROCESS Model 4 (simple mediation)")
    print(f"  Covars   : {covs if covs else 'none'}")
    print(f"  Indirect : b = {est:+.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
    print(f"  Result   : {sig}")
    if log_key:
        results_log.append({
            'Test': log_key, 'b': round(est, 3), 'p': np.nan,
            'sig': '—', 'R2': np.nan,
            'CI': f'[{lo:.3f}, {hi:.3f}]', 'Supported': supported
        })
    return est, lo, hi

def bootstrap_serial(data, x, m1, m2, y, covs=None, n_boot=5000, ci=95, label="", log_key=None):
    """Serial mediation — PROCESS Model 6: x -> m1 -> m2 -> y"""
    np.random.seed(42)
    indirect = []
    cs = (' + ' + ' + '.join(covs)) if covs else ''
    for _ in range(n_boot):
        s = data.sample(len(data), replace=True)
        a = smf.ols(f'{m1} ~ {x}', data=s).fit().params[x]
        b = smf.ols(f'{m2} ~ {m1} + {x}', data=s).fit().params[m1]
        c = smf.ols(f'{y} ~ {m2} + {m1} + {x}{cs}', data=s).fit().params[m2]
        indirect.append(a * b * c)
    indirect = np.array(indirect)
    alpha_pct = (100 - ci) / 2
    lo, hi = np.percentile(indirect, [alpha_pct, 100 - alpha_pct])
    est = np.mean(indirect)
    supported = not (lo <= 0 <= hi)
    sig = "SUPPORTED ✓" if supported else "NOT SUPPORTED ✗ (CI includes 0)"
    print(f"\n  Testing  : {label}")
    print(f"  Model    : PROCESS Model 6 (serial mediation)")
    print(f"  Covars   : {covs if covs else 'none'}")
    print(f"  Indirect : b = {est:+.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
    print(f"  Result   : {sig}")
    if log_key:
        results_log.append({
            'Test': log_key, 'b': round(est, 3), 'p': np.nan,
            'sig': '—', 'R2': np.nan,
            'CI': f'[{lo:.3f}, {hi:.3f}]', 'Supported': supported
        })
    return est, lo, hi


# ─────────────────────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("ANALYSIS SCRIPT v7 — Configurable Trust Variables")
print("=" * 70)
print(f"  Goodwill trust variable  : {LABEL_GOODWILL}")
print(f"  Qual trust variable      : {LABEL_QUAL}")
print(f"  Overall trust            : {LABEL_OVERALL}")
print(f"  Chatbot n = {sum(df['condition'] == 'Chatbot')}  |  ECA n = {sum(df['condition'] == 'ECA')}  |  Total n = {len(df)}")

print("\nDescriptive Statistics (trust + outcome variables):")
desc_cols = [VAR_OVERALL, VAR_GOODWILL, VAR_QUAL,
             'health_rating', 'behavioral_intention', 'continuance_intention']
print(df[desc_cols].rename(columns={
    VAR_OVERALL: 'delta_overall', VAR_GOODWILL: LABEL_GOODWILL, VAR_QUAL: LABEL_QUAL
}).describe().round(3).to_string())

print("\nCovariate map:")
cov_display = {
    LABEL_GOODWILL: COVS[VAR_GOODWILL] or ['none'],
    LABEL_QUAL:     COVS[VAR_QUAL] or ['none'],
    'delta_overall':         COVS[VAR_OVERALL] or ['none'],
    **{k: v or ['none'] for k, v in COVS.items() if k in ALL_OUTCOMES}
}
for k, v in cov_display.items():
    print(f"  {k:<35} : {v}")





# ─────────────────────────────────────────────────────────────
# H1 — CONDITION -> OVERALL TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H1 — Condition → Overall Trust Loss",
        f"Covariates: {COVS[VAR_OVERALL] or 'none'}")

run_ols(
    f'{VAR_OVERALL} ~ cond_dummy{cov_str(VAR_OVERALL)}',
    df, 'cond_dummy',
    f"ECA condition predicts less overall trust loss [{LABEL_OVERALL}]",
    log_key='H1: condition -> delta_overall'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA'][VAR_OVERALL].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot'][VAR_OVERALL].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# H1a — CONDITION -> GOODWILL TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H1a — Condition → Goodwill Trust Loss",
        f"Variable: {LABEL_GOODWILL}  |  Covariates: {COVS[VAR_GOODWILL] or 'none'}")

run_ols(
    f'{VAR_GOODWILL} ~ cond_dummy{cov_str(VAR_GOODWILL)}',
    df, 'cond_dummy',
    f"ECA condition predicts less goodwill trust loss [{LABEL_GOODWILL}]",
    log_key=f'H1a: condition -> {LABEL_GOODWILL}'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA'][VAR_GOODWILL].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot'][VAR_GOODWILL].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# H1b — CONDITION -> COMPETENCE TRUST LOSS 
# ─────────────────────────────────────────────────────────────

section("H1b — Condition → Competence Trust Loss",
        f"Variable: {LABEL_QUAL}  |  Covariates: {COVS[VAR_QUAL] or 'none'}")

run_ols(
    f'{VAR_QUAL} ~ cond_dummy{cov_str(VAR_QUAL)}',
    df, 'cond_dummy',
    f"ECA condition predicts less competence trust loss [{LABEL_QUAL}]",
    log_key=f'H1b: condition -> {LABEL_QUAL}'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA'][VAR_QUAL].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot'][VAR_QUAL].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# H2 — GOODWILL TRUST LOSS -> QUALIFICATION TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H2 — Goodwill Trust Loss → Qualification Trust Loss",
        f"{LABEL_GOODWILL} → {LABEL_QUAL}  |  Covariates: {COVS[VAR_QUAL] or 'none'}")

run_ols(
    f'{VAR_QUAL} ~ {VAR_GOODWILL}{cov_str(VAR_QUAL)}',
    df, VAR_GOODWILL,
    f"{LABEL_GOODWILL} predicts {LABEL_QUAL}",
    log_key=f'H2: {LABEL_GOODWILL} -> {LABEL_QUAL}'
)

print(f"\n  Relationship between {LABEL_GOODWILL} and {LABEL_QUAL}: "
      f"r = {df[[VAR_GOODWILL, VAR_QUAL]].dropna().corr().iloc[0,1]:.3f}")

# ─────────────────────────────────────────────────────────────
# DIRECT EFFECTS — TRUST LOSS -> OUTCOMES
# ─────────────────────────────────────────────────────────────

section("DIRECT EFFECTS — Trust Loss Dimensions → Outcomes",
        "Covariates vary by outcome (see covariate map above)")

#, 'behavioral_intention', 'continuance_intention'
for outcome in ['health_rating']:
    subsection(f"Outcome: {outcome}  |  Covariates: {COVS[outcome]}")

    run_ols(
        f'{outcome} ~ {VAR_OVERALL}{cov_str(outcome)}',
        df, VAR_OVERALL,
        f"Overall trust loss [{LABEL_OVERALL}] → {outcome}",
        log_key=f'Direct: overall -> {outcome}'
    )
    run_ols(
        f'{outcome} ~ {VAR_GOODWILL}{cov_str(outcome)}',
        df, VAR_GOODWILL,
        f"{LABEL_GOODWILL} → {outcome}",
        log_key=f'Direct: {LABEL_GOODWILL} -> {outcome}'
    )
    run_ols(
        f'{outcome} ~ {VAR_QUAL}{cov_str(outcome)}',
        df, VAR_QUAL,
        f"{LABEL_QUAL} → {outcome}",
        log_key=f'Direct: {LABEL_QUAL} -> {outcome}'
    )


# ─────────────────────────────────────────────────────────────
# HEALTH RATING -> BEHAVIORAL INTENTION 
# ─────────────────────────────────────────────────────────────
run_ols(
    f'behavioral_intention ~ health_rating{cov_str("behavioral_intention")}',
    df, 'health_rating',
    "Health rating directly predicts behavioral intention",
    log_key='Direct: health_rating -> behavioral_intention'
)


# ─────────────────────────────────────────────────────────────
# BEHAVIORAL INTENTION -> CONTINUANCE INTENTION
# ─────────────────────────────────────────────────────────────

section("Behavioral Intention → Continuance Intention",
        f"Covariates: {COVS['continuance_intention']}")

run_ols(
    f'continuance_intention ~ behavioral_intention{cov_str("continuance_intention")}',
    df, 'behavioral_intention',
    "Behavioral intention directly predicts continuance intention",
    log_key='Behavioral Intention → Continuance Intention'
)


# ─────────────────────────────────────────────────────────────
#  SIMPLE MEDIATION (PROCESS Model 4)
# ─────────────────────────────────────────────────────────────
# H3 — SIMPLE MEDIATION VIA HEALTH RATING → TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H3 — Simple Mediation via Health Rating",
        f"condition → health_rating → {LABEL_OVERALL}  |  PROCESS Model 4")

for input_var in ['health_rating']:
    subsection(f"Input: {input_var}  |  Covariates: {COVS[input_var]}")

    m_d = smf.ols(f'{VAR_OVERALL} ~ cond_dummy{cov_str(input_var)}', data=df).fit()
    print_result(
        f"Direct effect: condition → {VAR_OVERALL}",
        m_d.params['cond_dummy'], m_d.pvalues['cond_dummy'],
        log_key=f'H3 direct: condition -> {VAR_OVERALL}'
    )

    bootstrap_mediation(
        df, 'cond_dummy', input_var, VAR_OVERALL,
        covs=COVS[input_var],
        label=f"Indirect: condition → {input_var} → {VAR_OVERALL}",
        log_key=f'H3 indirect: cond -> {input_var} -> overall'
    )


# ─────────────────────────────────────────────────────────────
# H3a — SIMPLE MEDIATION VIA GOODWILL TRUST LOSS (PROCESS Model 4)
# ─────────────────────────────────────────────────────────────

section("H3a — Simple Mediation via Goodwill Trust Loss",
        f"condition → input → {LABEL_GOODWILL}  |  PROCESS Model 4")

for input_var in ['health_rating']:
    subsection(f"Input: {input_var}  |  Covariates: {COVS[input_var]}")

    bootstrap_mediation(
        df, 'cond_dummy', input_var, VAR_GOODWILL,
        covs=COVS[input_var],
        label=f"Indirect: condition → {input_var} → {LABEL_GOODWILL}",
        log_key=f'H3a indirect: cond -> {input_var} -> {LABEL_GOODWILL}'
    )


# ─────────────────────────────────────────────────────────────
# H3b — SIMPLE MEDIATION VIA QUALIFICATION TRUST (PROCESS Model 4)
# ─────────────────────────────────────────────────────────────

section("H3b — Simple Mediation via Qualification Trust",
        f"condition → input → {LABEL_QUAL}  |  PROCESS Model 4")

for input_var in ['health_rating']:
    subsection(f"Input: {input_var}  |  Covariates: {COVS[input_var]}")

    bootstrap_mediation(
        df, 'cond_dummy', input_var, VAR_QUAL,
        covs=COVS[input_var],
        label=f"Indirect: condition → {input_var} → {LABEL_QUAL}",
        log_key=f'H3b indirect: cond -> {input_var} -> {LABEL_QUAL}'
    )


# ─────────────────────────────────────────────────────────────
# RESULTS SUMMARY TABLE
# ─────────────────────────────────────────────────────────────

section("RESULTS SUMMARY",
        f"Trust variables used — goodwill: {LABEL_GOODWILL}  |  qual: {LABEL_QUAL}")

summary_df = pd.DataFrame(results_log)
summary_df['Supported'] = summary_df['Supported'].map({True: '✓', False: '✗'})
summary_df['CI']        = summary_df['CI'].fillna('—')
summary_df = summary_df.reindex(columns=['Test', 'b', 'p', 'sig', 'CI', 'R2', 'Supported'])

print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

ANALYSIS SCRIPT v7 — Configurable Trust Variables
  Goodwill trust variable  : delta_goodwill
  Qual trust variable      : delta_competence
  Overall trust            : overall (delta_goodwill + delta_competence)
  Chatbot n = 29  |  ECA n = 27  |  Total n = 56

Descriptive Statistics (trust + outcome variables):
       delta_overall  delta_goodwill  delta_competence  health_rating  behavioral_intention  continuance_intention
count         56.000          56.000            56.000         56.000                56.000                 56.000
mean           0.460           0.372             0.548          4.393                 4.827                  4.827
std            1.084           0.889             1.429          1.702                 1.844                  1.962
min           -1.083          -1.167            -2.333          1.000                 1.000                  1.000
25%           -0.042          -0.208             0.000          3.000                 3.917                  3

In [16]:
# ─────────────────────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("ANALYSIS SCRIPT v8 — Configurable Trust Variables")
print("=" * 70)
print(f"  Goodwill trust variable  : {LABEL_GOODWILL}")
print(f"  Qual trust variable      : {LABEL_QUAL}")
print(f"  Overall trust            : {LABEL_OVERALL}")
print(f"  Chatbot n = {sum(df['condition'] == 'Chatbot')}  |  ECA n = {sum(df['condition'] == 'ECA')}  |  Total n = {len(df)}")

print("\nDescriptive Statistics (trust + outcome variables):")
desc_cols = [VAR_OVERALL, VAR_GOODWILL, VAR_QUAL,
             'health_rating', 'behavioral_intention', 'continuance_intention']
print(df[desc_cols].rename(columns={
    VAR_OVERALL: 'delta_overall', VAR_GOODWILL: LABEL_GOODWILL, VAR_QUAL: LABEL_QUAL
}).describe().round(3).to_string())

print("\nCovariate map:")
cov_display = {
    LABEL_GOODWILL: COVS[VAR_GOODWILL] or ['none'],
    LABEL_QUAL:     COVS[VAR_QUAL] or ['none'],
    'delta_overall':         COVS[VAR_OVERALL] or ['none'],
    **{k: v or ['none'] for k, v in COVS.items() if k in ALL_OUTCOMES}
}
for k, v in cov_display.items():
    print(f"  {k:<35} : {v}")


# ─────────────────────────────────────────────────────────────
# H1 — CONDITION -> OVERALL TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H1 — Condition → Overall Trust Loss",
        f"Covariates: {COVS[VAR_OVERALL] or 'none'}")

run_ols(
    f'{VAR_OVERALL} ~ cond_dummy{cov_str(VAR_OVERALL)}',
    df, 'cond_dummy',
    f"ECA condition predicts less overall trust loss [{LABEL_OVERALL}]",
    log_key='H1: condition -> delta_overall'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA'][VAR_OVERALL].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot'][VAR_OVERALL].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# H1a — CONDITION -> GOODWILL TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H1a — Condition → Goodwill Trust Loss",
        f"Variable: {LABEL_GOODWILL}  |  Covariates: {COVS[VAR_GOODWILL] or 'none'}")

run_ols(
    f'{VAR_GOODWILL} ~ cond_dummy{cov_str(VAR_GOODWILL)}',
    df, 'cond_dummy',
    f"ECA condition predicts less goodwill trust loss [{LABEL_GOODWILL}]",
    log_key='H1a: condition -> delta_goodwill'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA'][VAR_GOODWILL].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot'][VAR_GOODWILL].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# H1b — CONDITION -> QUALIFICATION TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H1b — Condition → Qualification Trust Loss",
        f"Variable: {LABEL_QUAL}  |  Covariates: {COVS[VAR_QUAL] or 'none'}")

run_ols(
    f'{VAR_QUAL} ~ cond_dummy{cov_str(VAR_QUAL)}',
    df, 'cond_dummy',
    f"ECA condition predicts less qualification trust loss [{LABEL_QUAL}]",
    log_key='H1b: condition -> delta_qualification'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA'][VAR_QUAL].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot'][VAR_QUAL].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# H2 — GOODWILL TRUST LOSS -> QUALIFICATION TRUST LOSS
# ─────────────────────────────────────────────────────────────

section("H2 — Goodwill Trust Loss → Qualification Trust Loss",
        f"{LABEL_GOODWILL} → {LABEL_QUAL}  |  Covariates: {COVS[VAR_QUAL] or 'none'}")

run_ols(
    f'{VAR_QUAL} ~ {VAR_GOODWILL}{cov_str(VAR_QUAL)}',
    df, VAR_GOODWILL,
    f"{LABEL_GOODWILL} predicts {LABEL_QUAL}",
    log_key='H2: delta_goodwill -> delta_qualification'
)

print(f"\n  Relationship between {LABEL_GOODWILL} and {LABEL_QUAL}: "
      f"r = {df[[VAR_GOODWILL, VAR_QUAL]].dropna().corr().iloc[0,1]:.3f}")


# ─────────────────────────────────────────────────────────────
# A-PATH — CONDITION -> HEALTH RATING (mediator path for H3)
# ─────────────────────────────────────────────────────────────

section("A-path — Condition → Health Rating",
        f"Required for H3 mediation  |  Covariates: {COVS['health_rating']}")

run_ols(
    f'health_rating ~ cond_dummy{cov_str("health_rating")}',
    df, 'cond_dummy',
    "ECA condition predicts inflated health rating",
    log_key='A-path: condition -> health_rating'
)
print(f"\n  M (ECA) = {df[df['condition']=='ECA']['health_rating'].mean():.3f}  |  "
      f"M (Chatbot) = {df[df['condition']=='Chatbot']['health_rating'].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# B-PATH — HEALTH RATING -> TRUST LOSS DIMENSIONS
# ─────────────────────────────────────────────────────────────

section("B-path — Health Rating → Trust Loss",
        "Required for H3 mediation  |  Covariates vary by outcome")

run_ols(
    f'{VAR_OVERALL} ~ health_rating{cov_str(VAR_OVERALL)}',
    df, 'health_rating',
    f"Health rating → overall trust loss [{LABEL_OVERALL}]",
    log_key='B-path: health_rating -> delta_overall'
)
run_ols(
    f'{VAR_GOODWILL} ~ health_rating{cov_str(VAR_GOODWILL)}',
    df, 'health_rating',
    f"Health rating → {LABEL_GOODWILL}",
    log_key='B-path: health_rating -> delta_goodwill'
)
run_ols(
    f'{VAR_QUAL} ~ health_rating{cov_str(VAR_QUAL)}',
    df, 'health_rating',
    f"Health rating → {LABEL_QUAL}",
    log_key='B-path: health_rating -> delta_qualification'
)


# ─────────────────────────────────────────────────────────────
# H3 — MEDIATION: condition → health_rating → overall trust loss
# ─────────────────────────────────────────────────────────────

section("H3 — Mediation via Health Rating → Overall Trust Loss",
        f"condition → health_rating → {LABEL_OVERALL}  |  PROCESS Model 4")

m_d = smf.ols(f'{VAR_OVERALL} ~ cond_dummy{cov_str(VAR_OVERALL)}', data=df).fit()
print_result(
    f"Total effect: condition → {VAR_OVERALL}",
    m_d.params['cond_dummy'], m_d.pvalues['cond_dummy'],
    log_key='H3 total: condition -> delta_overall'
)

bootstrap_mediation(
    df, 'cond_dummy', 'health_rating', VAR_OVERALL,
    covs=COVS['health_rating'],
    label=f"Indirect: condition → health_rating → {VAR_OVERALL}",
    log_key='H3 indirect: cond -> health_rating -> delta_overall'
)


# ─────────────────────────────────────────────────────────────
# H3a — MEDIATION: condition → health_rating → goodwill trust loss
# ─────────────────────────────────────────────────────────────

section("H3a — Mediation via Health Rating → Goodwill Trust Loss",
        f"condition → health_rating → {LABEL_GOODWILL}  |  PROCESS Model 4")

bootstrap_mediation(
    df, 'cond_dummy', 'health_rating', VAR_GOODWILL,
    covs=COVS['health_rating'],
    label=f"Indirect: condition → health_rating → {LABEL_GOODWILL}",
    log_key='H3a indirect: cond -> health_rating -> delta_goodwill'
)


# ─────────────────────────────────────────────────────────────
# H3b — MEDIATION: condition → health_rating → qualification trust loss
# ─────────────────────────────────────────────────────────────

section("H3b — Mediation via Health Rating → Qualification Trust Loss",
        f"condition → health_rating → {LABEL_QUAL}  |  PROCESS Model 4")

bootstrap_mediation(
    df, 'cond_dummy', 'health_rating', VAR_QUAL,
    covs=COVS['health_rating'],
    label=f"Indirect: condition → health_rating → {LABEL_QUAL}",
    log_key='H3b indirect: cond -> health_rating -> delta_qualification'
)


# ─────────────────────────────────────────────────────────────
# H4 — HEALTH RATING -> BEHAVIORAL INTENTION
# ─────────────────────────────────────────────────────────────

section("H4 — Health Rating → Behavioral Intention",
        f"Covariates: {COVS['behavioral_intention']}")

run_ols(
    f'behavioral_intention ~ health_rating{cov_str("behavioral_intention")}',
    df, 'health_rating',
    "Health rating predicts behavioral intention to follow recommendations",
    log_key='H4: health_rating -> behavioral_intention'
)


# ─────────────────────────────────────────────────────────────
# H5 — BEHAVIORAL INTENTION -> CONTINUANCE INTENTION
# ─────────────────────────────────────────────────────────────

section("H5 — Behavioral Intention → Continuance Intention",
        f"Covariates: {COVS['continuance_intention']}")

run_ols(
    f'continuance_intention ~ behavioral_intention{cov_str("continuance_intention")}',
    df, 'behavioral_intention',
    "Behavioral intention predicts continuance intention",
    log_key='H5: behavioral_intention -> continuance_intention'
)


# ─────────────────────────────────────────────────────────────
# EXPLORATORY — TRUST LOSS -> BEHAVIORAL INTENTION
# ─────────────────────────────────────────────────────────────

section("Exploratory — Trust Loss → Behavioral Intention",
        f"Covariates: {COVS['behavioral_intention']}")

run_ols(
    f'behavioral_intention ~ {VAR_OVERALL}{cov_str("behavioral_intention")}',
    df, VAR_OVERALL,
    f"Overall trust loss [{LABEL_OVERALL}] → behavioral intention",
    log_key='Exploratory: delta_overall -> behavioral_intention'
)
run_ols(
    f'behavioral_intention ~ {VAR_GOODWILL}{cov_str("behavioral_intention")}',
    df, VAR_GOODWILL,
    f"{LABEL_GOODWILL} → behavioral intention",
    log_key='Exploratory: delta_goodwill -> behavioral_intention'
)
run_ols(
    f'behavioral_intention ~ {VAR_QUAL}{cov_str("behavioral_intention")}',
    df, VAR_QUAL,
    f"{LABEL_QUAL} → behavioral intention",
    log_key='Exploratory: delta_qualification -> behavioral_intention'
)

ANALYSIS SCRIPT v8 — Configurable Trust Variables
  Goodwill trust variable  : delta_goodwill
  Qual trust variable      : delta_competence
  Overall trust            : overall (delta_goodwill + delta_competence)
  Chatbot n = 29  |  ECA n = 27  |  Total n = 56

Descriptive Statistics (trust + outcome variables):
       delta_overall  delta_goodwill  delta_competence  health_rating  behavioral_intention  continuance_intention
count         56.000          56.000            56.000         56.000                56.000                 56.000
mean           0.460           0.372             0.548          4.393                 4.827                  4.827
std            1.084           0.889             1.429          1.702                 1.844                  1.962
min           -1.083          -1.167            -2.333          1.000                 1.000                  1.000
25%           -0.042          -0.208             0.000          3.000                 3.917                  3

(<statsmodels.regression.linear_model.RegressionResultsWrapper at 0x16c9b65d0>,
 np.float64(0.3973266706615723),
 np.float64(-0.9310974322512392),
 np.float64(-0.31533899307506086))